# 03 — Screening, watchlists, and probability calibration

This notebook shows the current discovery-to-calibration workflow. The recommended high-level path is:

1. screen a universe for candidate baskets;
2. save selected baskets as a persistent watchlist;
3. generate leakage-safe historical evidence/outcome records for every basket;
4. pool those records into a transparent probability calibration;
5. link that calibration into `portfolio.json`.

The lower-level single-basket calibration functions remain available, but the watchlist-level workflow is now the preferred route for live Cobasket use.


In [ ]:
from cobasket import calibrate_watchlist
from cobasket.evidence import BasketWatchlist, save_probability_calibration
from cobasket.workflow import PortfolioConfig


## 1. Start from a screened watchlist

The screen can create this file directly:

```bash
cobasket-screen --period 5y --top-n 20 --watchlist-out screened_watchlist.json
```

The screen proposes related groups, applies the Johansen threshold, and preliminary-backtests the survivors. This is target selection, not proof of future profitability.


In [ ]:
watchlist = BasketWatchlist.load("../screened_watchlist.json")
watchlist


## 2. Define the portfolio configuration

The watchlist says what remains under observation. Holdings say what is currently owned. A zero holding can remain on the watchlist and later become eligible for a buy recommendation.


In [ ]:
config = PortfolioConfig(
    holdings={},
    cash=10_000.0,
    watchlist_path="screened_watchlist.json",
    calibration_path=None,
    period="5y",
    z_window=60,
    min_trace_ratio=1.0,
)
config.save("../portfolio.json")


## 3. Fit a watchlist-level probability calibration

At each historical evaluation date Cobasket fits each basket using only earlier prices, calculates the evidence score, then measures what happened over the following horizon. Records from different baskets are pooled only after those leakage-safe evaluations are generated.

`horizon=20` is roughly one trading month. `step=5` evaluates approximately weekly, so adjacent outcomes overlap. Notebook 10 diagnoses the consequences of that overlap.


In [ ]:
result = calibrate_watchlist(
    "../portfolio.json",
    train_window=252,
    horizon=20,
    step=5,
)
result.basket_summary


In [ ]:
result.calibration.table


The probability is **relative**: it estimates how often a ticker with a similar score outperformed the equal-weight return of its own basket over the forecast horizon. It is not the probability that the ticker's absolute price rises.


## 4. Save the calibration and diagnostic records

The calibration JSON is used by live reports. The Parquet records are used by Notebook 10 for sign, per-basket, and overlap diagnostics.


In [ ]:
save_probability_calibration(result.calibration, "../probability_calibration.json")
result.records.to_parquet("../calibration_records.parquet", index=False)


For everyday use, the equivalent one-command workflow is:

```bash
cobasket-calibrate \
    --portfolio portfolio.json \
    --output probability_calibration.json \
    --records-out calibration_records.parquet \
    --train-window 252 \
    --horizon 20 \
    --step 5 \
    --update-portfolio
```

`--update-portfolio` writes the calibration path into `portfolio.json`, after which `cobasket-report` and the GUI use calibrated recommendations automatically.


## 5. Validate before trusting the probabilities

Check the sample count in each score bin, per-basket behaviour, non-overlapping horizons, and sign invariance. Notebook 10 performs those diagnostics.
